# Decision-Guided Patch Search — live demo (Colab, self-contained)

Two models from Hugging Face, one GPU runtime, no cloud accounts:

| Role | Model | Size |
|---|---|---|
| **Proposer** (writes K candidate fixes) | `Qwen/Qwen2.5-Coder-3B-Instruct` | ~6 GB |
| **Ranker** (scores candidates: Strands Decider 2B) | `StrandsAgents/strands-decider-2B-qwen3.5-v1-2610` | ~4 GB |

Loop: **propose → Decider ranks → run real tests in ranked order → stop at first pass.**
Then the same tasks run *unranked* to show what the ranking saves.

**Runtime:** GPU (Runtime → Change runtime type → T4 or better). Run top to bottom.
First run downloads ~10 GB of weights — up to 15 min.

In [ ]:
# Cell 1 — install
!pip install -q strands-decider torchao==0.16.0 transformers accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cell 2 — project code from GitHub (or uploaded zip fallback).
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/pkpkp-pk/aws"
PROJECT_DIR = "/content/decider-patch-search"

if REPO_URL:
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)   # clear stale/partial copies
    subprocess.run(["git", "clone", "-q", REPO_URL, PROJECT_DIR], check=True)
elif not os.path.exists(PROJECT_DIR):
    zips = glob.glob("/content/*.zip")
    assert zips, "set REPO_URL or upload the project zip to /content"
    subprocess.run(["unzip", "-q", "-o", zips[0], "-d", "/content"], check=True)
    PROJECT_DIR = os.path.splitext(zips[0])[0]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT_DIR],
               check=True)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import agent.search, decider_ranker.ranker, eval.tasks
print("project importable from", PROJECT_DIR)

In [ ]:
# Cell 3 — start the Decider server (same runtime, no tunnel needed).
# First run downloads ~4GB of weights — up to 15 min.
import json, subprocess, time, urllib.request

proc = subprocess.Popen(
    ['strands-decider', 'serve',
     'StrandsAgents/strands-decider-2B-qwen3.5-v1-2610',
     '--device', 'cuda', '--port', '8099'],
    stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
print('server pid', proc.pid)

for i in range(180):
    if proc.poll() is not None:
        print('SERVER DIED'); print(open('/content/server.log').read()[-3000:]); break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8099/health', timeout=2) as r:
            print('health:', json.loads(r.read())); break
    except Exception:
        time.sleep(5)
else:
    print('TIMEOUT'); print(open('/content/server.log').read()[-3000:])

In [ ]:
# Cell 4 — load the proposer model (replaces Amazon Bedrock in this demo).
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

PROPOSER = "Qwen/Qwen2.5-Coder-3B-Instruct"
tok = AutoTokenizer.from_pretrained(PROPOSER)
model = AutoModelForCausalLM.from_pretrained(
    PROPOSER, torch_dtype=torch.float16, device_map="cuda")
print("proposer loaded:", PROPOSER)

In [ ]:
# Cell 5 — wire the HF proposer into the project's proposer contract.
# Reuses the project's prompt and JSON extraction verbatim.
from agent.proposer import _PROMPT, _extract_patches, _fn_name

def hf_propose(source: str, failure: str, k: int = 8) -> list[str]:
    prompt = _PROMPT.format(source=source, failure=failure, k=k)
    msgs = [{"role": "user", "content": prompt}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k2: v.to("cuda") for k2, v in enc.items()}   # BatchEncoding-safe
    out = model.generate(**enc, max_new_tokens=2048, temperature=0.8,
                         do_sample=True, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][enc["input_ids"].shape[1]:],
                      skip_special_tokens=True)
    patches = _extract_patches(text, _fn_name(source))
    # dedupe
    seen, unique = set(), []
    import ast
    for p in patches:
        key = ast.dump(ast.parse(p))
        if key not in seen:
            seen.add(key); unique.append(p)
    print(f"proposed {len(patches)} patches, {len(unique)} unique")
    return unique

In [ ]:
# Cell 6 — LIVE: one task, Decider-ranked search, event by event.
from agent.search import search_fix
from decider_ranker.ranker import DeciderRanker
from eval.tasks import load_tasks

task = load_tasks("dev")[0]   # dev_clamp
print(f"task: {task.name} ({task.bug_class})")
print(task.source)

def show(ev):
    if ev["t"] == "ranked":
        print(f"  ranked scores: {ev['scores']}")
    elif ev["t"] == "ran":
        print(f"  rank {ev['rank']} (score {ev['score']:.3f}) -> "
              + ("PASS" if ev["passed"] else "fail"))

out = search_fix(task.source, task.tests, k=6, ranker=DeciderRanker(),
                 proposer=hf_propose, on_event=show)
print()
print("FIXED" if out.fixed else "NOT FIXED",
      f"| test runs: {out.test_runs} | decider: {out.decider_latency_ms:.0f} ms")
if out.fixed:
    print(out.fix.code)

In [ ]:
# Cell 7 — same candidates, unranked (proposal order = random-ish baseline).
out_blind = search_fix(task.source, task.tests, k=6, ranker=None,
                       proposer=hf_propose, on_event=show)
print()
print("FIXED" if out_blind.fixed else "NOT FIXED",
      f"| test runs: {out_blind.test_runs}")
print(f"ranking saved {out_blind.test_runs - out.test_runs} test runs on this task")

In [ ]:
# Cell 8 — mini benchmark: guided vs unranked over dev tasks.
from eval.tasks import load_tasks

rows = []
for t in load_tasks("dev")[:4]:
    guided = search_fix(t.source, t.tests, k=6, ranker=DeciderRanker(),
                        proposer=hf_propose)
    blind = search_fix(t.source, t.tests, k=6, ranker=None, proposer=hf_propose)
    rows.append((t.name,
                 guided.test_runs if guided.fixed else "FAIL",
                 blind.test_runs if blind.fixed else "FAIL"))
    print(f"{t.name:24s} guided={rows[-1][1]}  unranked={rows[-1][2]}")

wins = sum(1 for _, g, b in rows if g != 'FAIL' and (b == 'FAIL' or g < b))
print(f"\nguided search won or tied-win on {wins}/{len(rows)} tasks")

## What you just saw

- **Decider 2B** scored runtime-generated candidates — options that did not exist
  when any model was trained — in one request per candidate (four typed questions,
  shared prefix encoded once).
- The **search** spent test runs (the expensive verifier) in Decider's order.
- Unranked search spent more runs on the same candidates.

Full experiment (6 rankers, AUROC, calibration, bootstrap CIs, Pareto figures):
`python -m eval.build_pool && python -m eval.run_rankers --with decider` on the
repo. This notebook is the demo; the repo is the evidence.